In [ ]:
import pandas as pd
import numpy as np
import random
import string
from datetime import date, datetime

# -----------------------------
# PARAMETERS
# -----------------------------
start_date = date(2023, 1, 1)
end_date = date(2023, 12, 31)
date_range = pd.date_range(start_date, end_date)

num_accounts = 100000
regions = ['Domestic', 'EMEA', 'APAC', 'LATAM']
content_types = ['Series', 'Film']

# TARGET VALUES - Weekly HPS by region as specified
target_weekly_hps = {
    'Domestic': 8.0,   # Base value
    'EMEA': 5.2,       # 0.65 x Domestic
    'LATAM': 9.6,      # 1.2 x Domestic
    'APAC': 2.0        # 0.25 x Domestic
}

# Film to Series ratio - EXACT 1.4x
film_to_series_ratio = 1.4

# Distribution of accounts by region
region_distribution = {
    'Domestic': 0.35,
    'EMEA': 0.30,
    'LATAM': 0.20,
    'APAC': 0.15
}

# Content type distribution 
content_type_weights = {
    'Series': 0.7,  # 70% Series
    'Film': 0.3     # 30% Film
}

# Calculate exact hours per session needed
# Series is base, Film is 1.4x
series_hours_per_session = {}
film_hours_per_session = {}

for region in regions:
    # Calculate Series and Film targets to maintain exact 1.4x ratio
    # Using formula: Series * (0.7 + 0.3*1.4) = target
    series_base = target_weekly_hps[region] / 1.12
    film_base = series_base * film_to_series_ratio
    
    series_hours_per_session[region] = series_base
    film_hours_per_session[region] = film_base
    
    print(f"{region}: Series = {series_base:.2f}, Film = {film_base:.2f}, Ratio = {film_base/series_base:.2f}x")

# -----------------------------
# SEASONAL PATTERNS
# -----------------------------
# Monthly multipliers to enforce seasonal pattern
monthly_base_multiplier = {
    1: 0.70,   # January - lower
    2: 0.65,   # February - lowest
    3: 0.90,   # March - medium (spring break)
    4: 0.85,   # April - medium-low
    5: 0.80,   # May - lower
    6: 1.10,   # June - higher (school out)
    7: 1.30,   # July - highest (summer peak)
    8: 1.25,   # August - high (summer)
    9: 0.75,   # September - low (back to school)
    10: 0.85,  # October - medium-low
    11: 0.90,  # November - medium (Thanksgiving)
    12: 1.30   # December - highest (holidays)
}

# Region-specific seasonal adjustments
region_seasonal_adjustments = {
    'Domestic': {
        # Summer boost
        6: 1.2, 7: 1.3, 8: 1.2,
        # Holiday boost
        11: 1.2, 12: 1.3
    },
    'EMEA': {
        # Summer vacation
        7: 1.2, 8: 1.3,
        # Holiday period
        12: 1.2
    },
    'LATAM': {
        # Southern summer
        1: 1.3, 2: 1.3,
        # Holiday period
        12: 1.25
    },
    'APAC': {
        # New Year period
        1: 1.2, 2: 1.2,
        # Year-end
        12: 1.15
    }
}

# -----------------------------
# Generate Titles
# -----------------------------
# Top 5 Series titles - GUARANTEED TO BE TOP 5
top_5_series = [
    "Bluey Jr.",
    "Star Blasters",
    "Marvel Heroes United",
    "Frozen Tales Series",
    "Moana: Island Chronicles"
]

# Generate additional 45 Series titles
additional_series = [
    f"Series_{i+6}: {random.choice(['Chronicles of Aralon', 'Legends of Zephra', 'Tales of Noria', 'Heroes of Lumora', 'Knights of Eldoria'])}"
    for i in range(45)
]
series_titles = top_5_series + additional_series  # total = 50 series

# Generate Top 50 Film titles
top_25_films = [
    "Galactic Rebels",
    "Super Squirrel Squad",
    "Frozen Adventures",
    "Moana's Journey",
    "The Invincibles",
    "Star Wars: Rogue Universe",
    "Marvel Xtreme",
    "Pixar's Robot World",
    "The Lost Princess",
    "Zootropolis Returns",
    "Avatar 3",
    "The Lion Queen",
    "Toy Story 5",
    "Pirates of the Solar System",
    "Encanto 2",
    "Black Panther 3",
    "Guardians of the Multiverse",
    "Finding Marlin",
    "Aladdin: The New Adventure",
    "Beauty and the Dragon",
    "The Little Robot",
    "Jungle Cruise 2",
    "Wakanda Forever 2",
    "Inside Out 3",
    "Cars 4"
]
additional_films = [
    f"Film_{i+26}: {random.choice(['Quest of Dawn', 'Empire Rising', 'Shadow Hunters', 'Legends of Terra', 'Chronicles'])}"
    for i in range(25)
]
film_titles = top_25_films + additional_films  # total = 50 films

# Blockbusters for seasonal spikes
spring_film = "Galactic Rebels"       # March (Spring Break)
thanksgiving_film = "The Lost Princess" # November (Thanksgiving)
christmas_film = "Frozen Adventures"    # December (Christmas)

# Boost factors for title ranking - CRITICAL TO GET RIGHT
top_5_series_boost = 15.0   # Very high boost to ensure they're the top 5
other_series_boost = 0.2    # Very low boost so no other series cracks top 30
top_films_boost = 3.0       # Strong boost for ranks 6-30 (films)
other_films_boost = 0.8     # Lower boost for remaining films

# -----------------------------
# FUNCTIONS
# -----------------------------
def generate_account_id():
    """Generate a unique 20-character account ID."""
    return ''.join(random.choices(string.ascii_uppercase + string.digits, k=20))

# Pre-generate account IDs and assign regions
account_ids = [generate_account_id() for _ in range(num_accounts)]
account_regions = {}

# Distribute accounts by region according to weights
for account_id in account_ids:
    region = random.choices(
        regions, 
        weights=[region_distribution[r] for r in regions],
        k=1
    )[0]
    account_regions[account_id] = region

def get_seasonal_multiplier(date, region):
    """Get combined seasonal multiplier based on month and region."""
    # Base monthly multiplier
    month = date.month
    multiplier = monthly_base_multiplier[month]
    
    # Region-specific seasonal adjustment
    if month in region_seasonal_adjustments[region]:
        multiplier *= region_seasonal_adjustments[region][month]
    
    return multiplier

def get_blockbuster_factor(date, content_type, title):
    """Get blockbuster spike factor for special events."""
    if content_type != 'Film':
        return 1.0
        
    factor = 1.0
    
    # Spring Break (March)
    if date.month == 3 and title == spring_film:
        # Stronger spike across entire month
        factor *= 3.0
        # Extra spike during typical spring break (mid-March)
        if 10 <= date.day <= 25:
            factor *= 1.5
    
    # Thanksgiving (Nov 20-30)
    if date.month == 11 and date.day >= 20 and title == thanksgiving_film:
        factor *= 4.0
    
    # Christmas (Dec 15-31)
    if date.month == 12 and date.day >= 15 and title == christmas_film:
        factor *= 5.0
        # Extra boost for Christmas week
        if date.day >= 23:
            factor *= 1.5
            
    return factor

def get_title_boost(content_type, title):
    """Get boost factor based on content type and title."""
    if content_type == 'Series':
        if title in top_5_series:
            return top_5_series_boost
        else:
            return other_series_boost
    else:  # Film
        if title in top_25_films[:25]:  # Top 25 films
            return top_films_boost
        else:
            return other_films_boost

# -----------------------------
# DATA GENERATION STRATEGY
# -----------------------------
print("Generating streaming data...")
data = []
monthly_totals = {i: 0 for i in range(1, 13)}

# Efficient generation approach
for i, current_date in enumerate(date_range):
    if i % 30 == 0:
        print(f"Processing: {current_date.strftime('%Y-%m-%d')} - {((i+1)/len(date_range))*100:.1f}% complete")
    
    # Get monthly seasonal factor
    month = current_date.month
    
    # For each region, determine active accounts
    for region in regions:
        # Select a portion of accounts from this region
        region_accounts = [acc for acc, reg in account_regions.items() if reg == region]
        
        # Calculate active percentage based on seasonal factor
        base_active_pct = 0.1  # 10% base active rate
        seasonal_factor = get_seasonal_multiplier(current_date, region)
        active_pct = base_active_pct * seasonal_factor
        
        # Cap at reasonable maximum
        active_pct = min(active_pct, 0.25)  # Max 25% active in a day
        
        # Select active accounts
        active_count = int(len(region_accounts) * active_pct)
        active_accounts = random.sample(region_accounts, active_count)
        
        # Process each active account
        for account in active_accounts:
            # Determine sessions per day - most users have 1, some have 2
            sessions = 1 if random.random() < 0.8 else 2
            
            for _ in range(sessions):
                # Select content type using weighted distribution
                content_type = random.choices(
                    content_types, 
                    weights=[content_type_weights[ct] for ct in content_types],
                    k=1
                )[0]
                
                # Select title with weighted randomness
                if content_type == 'Series':
                    # Increased chance for top 5 series
                    if random.random() < 0.3:  # 30% chance for top 5
                        title = random.choice(top_5_series)
                    else:
                        title = random.choice(series_titles)
                else:
                    # Increased chance for top films
                    if random.random() < 0.8:  # 80% chance for top 25 films
                        title = random.choice(top_25_films)
                    else:
                        title = random.choice(film_titles)
                
                # Calculate base hours
                if content_type == 'Series':
                    base_hours = series_hours_per_session[region] / 7  # Convert weekly to daily
                else:
                    base_hours = film_hours_per_session[region] / 7  # Convert weekly to daily
                
                # Add random variation (±15%)
                random_factor = np.random.uniform(0.85, 1.15)
                
                # Apply modifiers
                title_boost = get_title_boost(content_type, title)
                blockbuster_factor = get_blockbuster_factor(current_date, content_type, title)
                
                # Calculate final hours
                session_hours = (
                    base_hours * random_factor * 
                    seasonal_factor * blockbuster_factor * 
                    title_boost
                )
                
                # Cap at reasonable maximum
                session_hours = min(session_hours, 5.0)
                
                # Add to data
                data.append([
                    current_date,
                    account,
                    region,
                    content_type,
                    title,
                    round(session_hours, 2)
                ])
                
                # Track monthly total
                monthly_totals[month] += session_hours

# -----------------------------
# CREATE DATAFRAME
# -----------------------------
df = pd.DataFrame(data, columns=[
    'business_date',
    'account_id',
    'region',
    'content_type',
    'full_title',
    'hours_streamed'
])

# -----------------------------
# DATA VALIDATION
# -----------------------------
print("\n--- VALIDATION ---")

# 1. Check dataset size
print(f"Total rows generated: {len(df):,}")

# 2. Check monthly distribution - show seasonality
monthly_data = df.groupby(df['business_date'].dt.month)['hours_streamed'].sum()
print("\nMonthly hours distribution:")
for month in range(1, 13):
    if month in monthly_data:
        print(f"Month {month}: {monthly_data[month]:,.0f} hours " + 
              f"({(monthly_data[month]/df['hours_streamed'].sum())*100:.1f}%)")

# 3. Check weekly HPS by region (December sample)
print("\nValidating December HPS...")
sample_week = df[(df['business_date'] >= '2023-12-24') & (df['business_date'] <= '2023-12-30')]
region_hours = sample_week.groupby('region')['hours_streamed'].sum()
region_accounts = sample_week.groupby('region')['account_id'].nunique()
region_hps = region_hours / region_accounts

print("\nWeekly HPS by Region (December):")
print("Region\tTarget HPS\tActual HPS\tRatio")
for region in regions:
    target = target_weekly_hps[region]
    actual = region_hps[region] if region in region_hps else 0
    ratio = actual / target if target > 0 else 0
    print(f"{region}\t{target:.1f}\t\t{actual:.1f}\t\t{ratio:.2f}x")

# 4. Check Film vs Series ratio
content_hours = sample_week.groupby(['region', 'content_type'])['hours_streamed'].sum()
content_accounts = sample_week.groupby(['region', 'content_type'])['account_id'].nunique()
content_hps = content_hours / content_accounts

print("\nFilm to Series Ratio by Region:")
for region in regions:
    if (region, 'Series') in content_hps and (region, 'Film') in content_hps:
        series_hps = content_hps[(region, 'Series')]
        film_hps = content_hps[(region, 'Film')]
        ratio = film_hps / series_hps
        print(f"{region}: Film = {film_hps:.1f}, Series = {series_hps:.1f}, Ratio = {ratio:.2f}x (Target: 1.40x)")

# 5. Check top titles ranking
title_hours = df.groupby('full_title')['hours_streamed'].sum().sort_values(ascending=False)
title_content_type = df.groupby('full_title')['content_type'].first()

print("\nTop 30 Titles:")
print("Rank\tTitle\t\t\tContent Type\tHours")
for i, (title, hours) in enumerate(title_hours.iloc[:30].items()):
    content = title_content_type[title]
    print(f"{i+1}\t{title[:22]}\t{content}\t\t{hours:,.0f}")

# Count Series vs Films in top 30
top5_series = sum(1 for title in title_hours.iloc[:5].index if title_content_type[title] == 'Series')
top5_films = sum(1 for title in title_hours.iloc[:5].index if title_content_type[title] == 'Film')

top6to30_series = sum(1 for title in title_hours.iloc[5:30].index if title_content_type[title] == 'Series')
top6to30_films = sum(1 for title in title_hours.iloc[5:30].index if title_content_type[title] == 'Film')

print("\nTop Title Distribution:")
print(f"Top 5: {top5_series} Series, {top5_films} Films")
print(f"Rank 6-30: {top6to30_series} Series, {top6to30_films} Films")

# Check if we need HPS calibration
needs_calibration = any(abs(region_hps.get(region, 0)/target_weekly_hps[region] - 1) > 0.1 
                      for region in regions)

# Check if we need film/series ratio calibration                  
needs_ratio_calibration = any(abs(content_hps.get((region, 'Film'), 0) / 
                              content_hps.get((region, 'Series'), 1) - film_to_series_ratio) > 0.1
                              for region in regions 
                              if (region, 'Series') in content_hps and (region, 'Film') in content_hps)

# Check if we need top titles calibration
needs_title_calibration = (top5_series != 5 or top6to30_series != 0)

# -----------------------------
# CALIBRATION (if needed)
# -----------------------------
if needs_calibration or needs_ratio_calibration or needs_title_calibration:
    print("\n--- APPLYING CALIBRATION ---")
    
    # 1. Region HPS Calibration
    if needs_calibration:
        hps_correction = {}
        for region in regions:
            if region in region_hps and region_hps[region] > 0:
                factor = target_weekly_hps[region] / region_hps[region]
                hps_correction[region] = factor
                print(f"HPS correction for {region}: {factor:.2f}x")
            else:
                hps_correction[region] = 1.0
    else:
        hps_correction = {region: 1.0 for region in regions}
    
    # 2. Film/Series Ratio Calibration
    if needs_ratio_calibration:
        ratio_correction = {}
        for region in regions:
            if (region, 'Series') in content_hps and (region, 'Film') in content_hps:
                series_hps = content_hps[(region, 'Series')]
                film_hps = content_hps[(region, 'Film')]
                current_ratio = film_hps / series_hps
                
                if abs(current_ratio - film_to_series_ratio) > 0.1:
                    # Calculate corrections
                    film_factor = film_to_series_ratio / current_ratio
                    
                    ratio_correction[(region, 'Film')] = film_factor
                    ratio_correction[(region, 'Series')] = 1.0  # Keep series as reference
                    
                    print(f"Film/Series ratio correction for {region}: Film *= {film_factor:.2f}")
                else:
                    ratio_correction[(region, 'Film')] = 1.0
                    ratio_correction[(region, 'Series')] = 1.0
            else:
                ratio_correction[(region, 'Film')] = 1.0
                ratio_correction[(region, 'Series')] = 1.0
    else:
        ratio_correction = {(region, content): 1.0 for region in regions for content in content_types}
    
    # 3. Title Ranking Calibration (if needed)
    if needs_title_calibration:
        print("Applying title boost calibration...")
    
    # Apply all calibrations
    def apply_calibration(row):
        region = row['region']
        content_type = row['content_type']
        title = row['full_title']
        hours = row['hours_streamed']
        
        # Region HPS correction
        region_factor = hps_correction.get(region, 1.0)
        
        # Content type ratio correction
        ratio_factor = ratio_correction.get((region, content_type), 1.0)
        
        # Title ranking correction (if needed)
        title_factor = 1.0
        if needs_title_calibration:
            if content_type == 'Series':
                if title in top_5_series:
                    # Boost top 5 series even more
                    title_factor = 2.0
                else:
                    # Reduce all other series
                    title_factor = 0.5
            elif content_type == 'Film' and title in top_25_films[:25]:
                # Boost top films slightly
                title_factor = 1.2
        
        # Combined factor
        combined_factor = region_factor * ratio_factor * title_factor
        
        return min(5.0, hours * combined_factor)
    
    # Apply calibration
    df['hours_streamed'] = df.apply(
        lambda row: round(apply_calibration(row), 2), 
        axis=1
    )
    
    print("Calibration applied. Verifying results...")
    
    # Recheck HPS and ratios with the same sample week
    sample_week = df[(df['business_date'] >= '2023-12-24') & (df['business_date'] <= '2023-12-30')]
    region_hours = sample_week.groupby('region')['hours_streamed'].sum()
    region_accounts = sample_week.groupby('region')['account_id'].nunique()
    region_hps = region_hours / region_accounts
    
    content_hours = sample_week.groupby(['region', 'content_type'])['hours_streamed'].sum()
    content_accounts = sample_week.groupby(['region', 'content_type'])['account_id'].nunique()
    content_hps = content_hours / content_accounts
    
    print("\nCalibrated Weekly HPS by Region:")
    for region in regions:
        target = target_weekly_hps[region]
        actual = region_hps[region] if region in region_hps else 0
        print(f"{region}: {actual:.1f} (Target: {target:.1f})")
    
    print("\nCalibrated Film to Series Ratio:")
    for region in regions:
        if (region, 'Series') in content_hps and (region, 'Film') in content_hps:
            series_hps = content_hps[(region, 'Series')]
            film_hps = content_hps[(region, 'Film')]
            ratio = film_hps / series_hps
            print(f"{region}: {ratio:.2f}x (Target: {film_to_series_ratio:.2f}x)")
    
    # Recheck top titles
    title_hours = df.groupby('full_title')['hours_streamed'].sum().sort_values(ascending=False)
    top5_series_after = sum(1 for title in title_hours.iloc[:5].index 
                          if title_content_type[title] == 'Series')
    top6to30_series_after = sum(1 for title in title_hours.iloc[5:30].index 
                              if title_content_type[title] == 'Series')
    
    print(f"\nCalibrated Title Ranking: Top 5 = {top5_series_after} Series, Rank 6-30 = {top6to30_series_after} Series")

# -----------------------------
# EXPORT CSV
# -----------------------------
file_path = 'disney_plus_streaming_data.csv'
df.to_csv(file_path, index=False)

print(f"\nCSV file saved at {file_path}")
print(f"Total rows generated: {len(df):,}")
print(f"Total hours: {df['hours_streamed'].sum():,.0f}")

Domestic: Series = 7.14, Film = 10.00, Ratio = 1.40x
EMEA: Series = 4.64, Film = 6.50, Ratio = 1.40x
APAC: Series = 1.79, Film = 2.50, Ratio = 1.40x
LATAM: Series = 8.57, Film = 12.00, Ratio = 1.40x
Generating streaming data...
Processing: 2023-01-01 - 0.3% complete
Processing: 2023-01-31 - 8.5% complete
Processing: 2023-03-02 - 16.7% complete
Processing: 2023-04-01 - 24.9% complete
Processing: 2023-05-01 - 33.2% complete
Processing: 2023-05-31 - 41.4% complete
Processing: 2023-06-30 - 49.6% complete
Processing: 2023-07-30 - 57.8% complete
Processing: 2023-08-29 - 66.0% complete
Processing: 2023-09-28 - 74.2% complete
Processing: 2023-10-28 - 82.5% complete
Processing: 2023-11-27 - 90.7% complete
Processing: 2023-12-27 - 98.9% complete

--- VALIDATION ---
Total rows generated: 4,515,998

Monthly hours distribution:
Month 1: 581,080 hours (5.5%)
Month 2: 471,205 hours (4.5%)
Month 3: 735,285 hours (7.0%)
Month 4: 650,886 hours (6.2%)
Month 5: 614,296 hours (5.8%)
Month 6: 1,033,508 hour